## Persiapan: Membuat SparkSession

Buat SparkSession sebagai lingkungan untuk menjalankan pengolahan data menggunakan PySpark serta mengimpor fungsi yang diperlukan untuk proses Join, Window Function, dan Spark SQL.

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, count, row_number
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Tugas5_JoinWindowSQL") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

26/09/17 10:13:37 WARN Utils: Your hostname, asih2 resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/09/17 10:13:37 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


SparkSession siap. Versi Spark: 3.5.6


In [3]:
import numpy as np
import pandas as pd

data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"]
}

df_target = spark.createDataFrame(
    pd.DataFrame(data_target_cabang)
)

df_transaksi = spark.read.csv(
    "hdfs://localhost:9000/user/mahasiswa/tugas5/transaksi_tugas5.csv",
    header=True,
    inferSchema=True
)

df_transaksi = df_transaksi.withColumn(
    "pendapatan",
    col("unit_terjual") * col("harga_satuan")
)

print("Data target:")
df_target.show()

print("Data transaksi:")
df_transaksi.show(5)

print("Jumlah transaksi:", df_transaksi.count())

Data target:


+----------+--------------+----------+
|      kota|target_bulanan|pic_cabang|
+----------+--------------+----------+
|  Magelang|      45000000|      Rani|
|Yogyakarta|      60000000|      Joko|
|  Semarang|      55000000|      Sari|
|      Solo|      40000000|      Bayu|
| Purworejo|      30000000|     Fitri|
+----------+--------------+----------+

Data transaksi:
+--------+--------------------+----------+------------+------------+----------+
|order_id|            kategori|      kota|unit_terjual|harga_satuan|pendapatan|
+--------+--------------------+----------+------------+------------+----------+
|   TRX-0|   Makanan & Minuman| Purworejo|           8|       75000|    600000|
|   TRX-1|          Elektronik|      Solo|           9|       75000|    675000|
|   TRX-2|Kesehatan & Kecan...|      Solo|           6|      100000|    600000|
|   TRX-3|             Fashion|Yogyakarta|           6|      100000|    600000|
|   TRX-4|          Elektronik|Yogyakarta|           2|       75000|    

## A. Join & Perbandingan Target

Hitung total `pendapatan` per `kota` dari `df_transaksi`, kemudian lakukan join dengan `df_target`. Tambahkan kolom `pencapaian_persen` untuk mengetahui persentase pencapaian pendapatan terhadap target bulanan. Urutkan hasil berdasarkan pencapaian tertinggi.

In [4]:
ringkasan_kota = df_transaksi.groupBy("kota").agg(
    spark_sum("pendapatan").alias("total_pendapatan")
)

hasil_A = ringkasan_kota.join(
    df_target,
    on="kota",
    how="inner"
)

hasil_A = hasil_A.withColumn(
    "pencapaian_persen",
    (col("total_pendapatan") / col("target_bulanan")) * 100
)

hasil_A.orderBy(
    col("pencapaian_persen").desc()
).show()

[Stage 11:>                                                         (0 + 4) / 4]

+----------+----------------+--------------+----------+------------------+
|      kota|total_pendapatan|target_bulanan|pic_cabang| pencapaian_persen|
+----------+----------------+--------------+----------+------------------+
| Purworejo|        45650000|      30000000|     Fitri|152.16666666666669|
|      Solo|        33475000|      40000000|      Bayu|           83.6875|
|Yogyakarta|        47275000|      60000000|      Joko| 78.79166666666667|
|  Magelang|        31650000|      45000000|      Rani| 70.33333333333334|
|  Semarang|        38175000|      55000000|      Sari|  69.4090909090909|
+----------+----------------+--------------+----------+------------------+



**Penjelasan:**  
Kode ini digunakan untuk menghitung total pendapatan setiap kota menggunakan `groupBy()` dan `spark_sum()`. Hasil tersebut kemudian digabungkan dengan `df_target` menggunakan `inner join` berdasarkan kolom `kota`. Setelah proses join, dibuat kolom `pencapaian_persen` dengan membandingkan total pendapatan terhadap target bulanan. Hasil akhirnya diurutkan dari persentase pencapaian tertinggi ke terendah.

## B. Window Function — Kategori Terlaris per Kota

Tentukan kategori dengan total pendapatan tertinggi pada setiap kota. Gunakan Window Function dan `row_number()` untuk memberikan peringkat kategori berdasarkan total pendapatan pada masing-masing kota. Tampilkan hanya kategori dengan peringkat pertama.

In [5]:
pendapatan_kategori = df_transaksi.groupBy(
    "kota", "kategori"
).agg(
    spark_sum("pendapatan").alias("total_pendapatan")
)

window_kategori = Window \
    .partitionBy("kota") \
    .orderBy(col("total_pendapatan").desc())

hasil_B = pendapatan_kategori.withColumn(
    "peringkat",
    row_number().over(window_kategori)
)

hasil_B.filter(
    col("peringkat") == 1
).orderBy("kota").show()

[Stage 12:>                                                         (0 + 1) / 1]

+----------+--------------------+----------------+---------+
|      kota|            kategori|total_pendapatan|peringkat|
+----------+--------------------+----------------+---------+
|  Magelang|Kesehatan & Kecan...|         7275000|        1|
| Purworejo|Kesehatan & Kecan...|        10075000|        1|
|  Semarang|        Rumah Tangga|        11125000|        1|
|      Solo|Kesehatan & Kecan...|         8425000|        1|
|Yogyakarta|             Fashion|        13325000|        1|
+----------+--------------------+----------------+---------+



**Penjelasan:**  
Kode ini digunakan untuk menghitung total pendapatan berdasarkan kota dan kategori. Selanjutnya, `Window.partitionBy("kota")` digunakan untuk membuat kelompok berdasarkan kota, sedangkan `orderBy()` mengurutkan kategori berdasarkan total pendapatan dari yang tertinggi. Fungsi `row_number()` memberikan peringkat pada setiap kategori dalam masing-masing kota. Dengan memfilter `peringkat == 1`, diperoleh kategori dengan total pendapatan tertinggi pada setiap kota.

## C. Spark SQL

Daftarkan `df_transaksi` dan `df_target` sebagai temporary view. Kemudian gunakan `spark.sql()` untuk menampilkan `kota`, `pic_cabang`, dan jumlah transaksi pada setiap kota. Urutkan hasil berdasarkan jumlah transaksi dari yang terbanyak.

In [6]:
df_transaksi.createOrReplaceTempView("transaksi")
df_target.createOrReplaceTempView("target_cabang")

print("Temporary view berhasil dibuat.")

Temporary view berhasil dibuat.


**Penjelasan:**  
Kode ini digunakan untuk mendaftarkan `df_transaksi` dan `df_target` sebagai temporary view dengan nama `transaksi` dan `target_cabang`. Temporary view memungkinkan kedua DataFrame digunakan seperti tabel dalam perintah SQL sehingga dapat diproses menggunakan `spark.sql()`.

### Menjalankan Kueri Spark SQL

Gunakan `spark.sql()` untuk menggabungkan data transaksi dengan data target berdasarkan `kota`, menghitung jumlah transaksi setiap kota, menampilkan PIC cabang, dan mengurutkan hasil berdasarkan jumlah transaksi terbanyak.

In [8]:
hasil_C = spark.sql("""
    SELECT
        t.kota,
        c.pic_cabang,
        COUNT(t.order_id) AS jumlah_transaksi
    FROM transaksi t
    JOIN target_cabang c
        ON t.kota = c.kota
    GROUP BY t.kota, c.pic_cabang
    ORDER BY jumlah_transaksi DESC
""")

hasil_C.show()

[Stage 19:===========================================>              (3 + 1) / 4]

+----------+----------+----------------+
|      kota|pic_cabang|jumlah_transaksi|
+----------+----------+----------------+
| Purworejo|     Fitri|             116|
|Yogyakarta|      Joko|             110|
|      Solo|      Bayu|              95|
|  Semarang|      Sari|              93|
|  Magelang|      Rani|              86|
+----------+----------+----------------+



**Penjelasan:**  
Kode ini menggunakan `spark.sql()` untuk menjalankan kueri SQL pada temporary view yang telah dibuat. Data transaksi digabungkan dengan data target berdasarkan kolom `kota`, kemudian `COUNT()` digunakan untuk menghitung jumlah transaksi setiap kota. `GROUP BY` digunakan untuk mengelompokkan data berdasarkan kota dan PIC cabang, sedangkan `ORDER BY` digunakan untuk mengurutkan hasil dari jumlah transaksi terbanyak ke paling sedikit.

## D. Kesimpulan

Berdasarkan hasil analisis pada bagian A dan B, jelaskan performa setiap cabang dengan menggunakan angka pendukung dari hasil analisis. Jelaskan pencapaian target dan kategori dengan pendapatan tertinggi pada setiap kota. Kesimpulan ditulis minimal 100 kata.

## Kesimpulan

Berdasarkan hasil analisis perbandingan target, setiap cabang memiliki tingkat pencapaian yang berbeda. Purworejo memiliki pencapaian target sebesar 152,17% dengan total pendapatan Rp45.650.000 dari target Rp30.000.000. Solo mencapai 83,69% dengan pendapatan Rp33.475.000 dari target Rp40.000.000, sedangkan Yogyakarta mencapai 78,79% dengan pendapatan Rp47.275.000 dari target Rp60.000.000. Magelang memiliki pencapaian sebesar 70,33% dengan pendapatan Rp31.650.000 dari target Rp45.000.000, dan Semarang mencapai 69,41% dengan pendapatan Rp38.175.000 dari target Rp55.000.000. Hasil Window Function menunjukkan bahwa kategori dengan pendapatan tertinggi di Magelang, Purworejo, dan Solo adalah Kesehatan & Kecantikan. Yogyakarta memiliki kategori terlaris Fashion, sedangkan Semarang memiliki kategori terlaris Rumah Tangga. Hasil ini menunjukkan adanya perbedaan performa penjualan dan kategori unggulan pada setiap cabang yang dapat digunakan sebagai bahan evaluasi strategi penjualan.